In [29]:
from langgraph.graph import add_messages, StateGraph, END, MessagesState
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.prebuilt import ToolNode
from langgraph.types import Command
from langgraph.checkpoint.memory import MemorySaver

In [30]:
Memory=MemorySaver()
load_dotenv()

True

In [31]:
search_tool=TavilySearchResults(
    max_results=5
)

tools=[search_tool]

In [32]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm_with_tool=llm.bind_tools(tools=tools)

prompt=ChatPromptTemplate([
    ("system","You are an AI assistant. Chat with the user in a very "
    "friendly and humanly way. Help the user and answer user queries. "
    "Only use tools if the answer to the user quries is not found "
    "in the chat history."),
    MessagesPlaceholder(variable_name="messages")
])

llm_chain=prompt | llm_with_tool

In [33]:
async def llm_node(state:MessagesState):
    result=await llm_chain.ainvoke(state)
    return {"messages":[result]}

async def tool_router(state:MessagesState):
    lastAIMessage=state["messages"][-1]
    if isinstance(lastAIMessage,AIMessage) and hasattr(lastAIMessage,"tool_calls") and len(lastAIMessage.tool_calls)>0:
        return "tool_executer"
    else:
        return END

In [34]:
graph=StateGraph(MessagesState)
graph.add_node("llm_executer",llm_node)
graph.add_node("tool_executer",ToolNode(tools=tools))

graph.add_edge("tool_executer","llm_executer")
graph.add_conditional_edges(
    "llm_executer",tool_router
)

graph.set_entry_point("llm_executer")

app=graph.compile(checkpointer=Memory)

config={
    "configurable":{
        "thread_id":1
    }
}

In [35]:
query="what is the climatic contition in malaysia today?"
async for event in app.astream({"messages":[HumanMessage(content=query)]}, config=config):
    print(event)

{'llm_executer': {'messages': [AIMessage(content=[{'type': 'text', 'text': "Hello there! I'd be happy to help you with that. Let me just check the current climatic conditions in Malaysia for you.", 'extras': {'signature': 'CsMDARFNMg/6Uc19XV2h+OMbff+Jl4I5elnlXdCpXH4O8z6OdhoWtFsTgaPpFh1Eb7is34ITYKMWCUiP2pDssVl092XMeMb3H5kbbnr9kJVJBIHe9zoyme+VtcNCn7XI+J1ttg2hJ/gPGIa1c9yFuqWet8wyQqi+BSNQWldXutLscO6k8+Xe9fa6qKhjzPuyeg8Sallo7gN7BXCuAg7owDQyjBDGy5CXyf/lrlWh6d3wcudt4GnTnbSYY9QzOmimPoRScbvvRGqxZ9/tfFZw50R0ZFN84gYkXZX9TV8K+jOspR8N7NTSmrnd8RP4QJAvkhlZS7I0w6+H4jwqSx2OTPp1m+asahhqLjh+752jlmb46OeNk1X4sNbFwL2eo+QSidIloMGm2vD6ztRijPJwMTpyUuWC3wgtP03iQmD8sPwsL9EUtoBiM3vbcP3hvnYtp2QVYq/lsLtY+OeRYKeGwyGtOzB59mUPMgb9ZN8W+NAVWDf87k8H9rYHDEbpOVPtre+oKfbOA0xLPmhVdy2EcgydT4hESYkJIKnU8wUbnRc8vljQ3JMerJy77NlDF4rflCxkr6Yig3vANNMS9WcQ05JZwrRn4g=='}}], additional_kwargs={'function_call': {'name': 'tavily_search_results_json', 'arguments': '{"query": "climatic condition in Malaysia today"}'}}, response_metadata={'